# pay roll

## payroll data type convertion

In [0]:
from pyspark.sql.functions import col,substring,to_date
df = spark.table("pl_workforce_catalog.bronze.payroll") \
     .withColumn("payroll_id", col("payroll_id").cast("int")) \
     .withColumn("employee_id", col("employee_id").cast("int")) \
     .withColumn("company_id", col("company_id").cast("int"))  \
     .withColumn("department_id", col("department_id").cast("int")) \
     .withColumn("pay_period_start",substring("pay_period_start", 0, 10)) \
     .withColumn("pay_period_start", to_date("pay_period_start","dd-MM-yyyy")) \
     .withColumn("pay_period_end",substring("pay_period_end", 0, 10)) \
     .withColumn("pay_period_end", to_date("pay_period_end","dd-MM-yyyy")) \
     .withColumn("pay_date",substring("pay_date", 0, 10)) \
     .withColumn("pay_date", to_date("pay_date","dd-MM-yyyy")) \
     .withColumn("gross_salary", col("gross_salary").cast("float")) \
     .withColumn("bonus", col("bonus").cast("float")) \
     .withColumn("overtime_pay", col("overtime_pay").cast("float")) \
     .withColumn("commission", col("commission").cast("float")) \
     .withColumn("allowances", col("allowances").cast("float")) \
     .withColumn("tax_deduction", col("tax_deduction").cast("float")) \
     .withColumn("social_security", col("social_security").cast("float")) \
     .withColumn("health_insurance", col("health_insurance").cast("float")) \
     .withColumn("retirement_contribution", col("retirement_contribution").cast("float")) \
     .withColumn("other_deductions", col("other_deductions").cast("float")) \
     .withColumn("net_salary", col("net_salary").cast("float")) 
display(df)

## filter wrong data 

In [0]:
df=df.filter(
    (col("gross_salary") != 0) &
    (col("pay_period_start") <= col("pay_period_end")))
   

## handling inconsistant data

In [0]:
from pyspark.sql.functions import col

employee_df = spark.table("pl_workforce_catalog.silver.employee").select("employee_id", "base_salary")

df = df.drop("gross_salary", "net_salary") \
    .join(employee_df, on="employee_id", how="left") \
    .withColumn(
        "gross_salary",
        col("base_salary") +
        col("bonus") +
        col("overtime_pay") +
        col("commission") +
        col("allowances")
    ) \
    .withColumn(
        "net_salary",
        col("gross_salary") -
        (
            col("tax_deduction") +
            col("social_security") +
            col("health_insurance") +
            col("retirement_contribution") +
            col("other_deductions")
        )
    ) \
    .drop("base_salary")
df.write.mode("overwrite").saveAsTable("pl_workforce_catalog.silver.payroll")
display(df.limit(10))